# Suspicious Login Attack Detector

This notebook implements a complete suspicious-login detection prototype:

1. Generate **3,000 synthetic rejected-login records** using 10 common email accounts.
2. Create chronological behavior features without future-data leakage.
3. Train a logistic-regression attack classifier.
4. Combine model probability with deterministic security rules.
5. Add a `thret` score from `0` to `1` and recommend an action.
6. Evaluate any new rejected-login attempt by calling one method.

> **Production warning:** The included labels and patterns are synthetic. Retrain and calibrate the model with confirmed real security outcomes before production use.


## 1. Environment

Required packages: `pandas`, `numpy`, `scikit-learn`, `joblib`, and `matplotlib`.

Uncomment the installation command only when the packages are missing.


In [57]:
# %pip install pandas numpy scikit-learn joblib matplotlib


## 2. Imports, schema, features, and thresholds

In [58]:
from __future__ import annotations
from sklearn.base import BaseEstimator, TransformerMixin

import inspect
import json
import math
import random
from collections import defaultdict, deque
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Iterable

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


BASE_COLUMNS = [
    "id",
    "email",
    "device_mac_id",
    "ip",
    "location",
    "time_to_attempt",
    "rejection_reason",
]

COMMON_EMAILS = [f"user{i}@example.com" for i in range(1, 11)]
LOCATIONS = [
    "Dhaka, BD",
    "Chattogram, BD",
    "Sylhet, BD",
    "Rajshahi, BD",
    "London, GB",
    "New York, US",
    "Singapore, SG",
    "Dubai, AE",
]
REJECTION_REASONS = [
    "invalid_password",
    "password_typo",
    "otp_failed",
    "expired_session",
    "unknown_device",
    "too_many_attempts",
    "blocked_ip",
]

NUMERIC_FEATURES = [
    "attempts_email_5m",
    "attempts_email_1h",
    "attempts_ip_5m",
    "attempts_ip_1h",
    "unique_emails_from_ip_10m",
    "unique_ips_for_email_1h",
    "unique_devices_for_email_24h",
    "seconds_since_email_attempt",
    "seconds_since_ip_attempt",
    "is_new_ip_for_email",
    "is_new_device_for_email",
    "is_new_location_for_email",
    "is_night",
    "is_weekend",
    "hour_sin",
    "hour_cos",
]
CATEGORICAL_FEATURES = ["location", "rejection_reason"]
MODEL_FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

ACTION_THRESHOLDS = {
    "medium_risk": 0.25,
    "otp_required": 0.40,
    "block_5_minutes": 0.54,
    "block_1_hour": 0.68,
    "temporary_block": 0.82,
    "permanent_block": 0.92,
}

from IPython.display import display
import matplotlib.pyplot as plt

## 3. Synthetic data generation

The generator creates normal user mistakes and four attack scenarios:

- Brute force against one account
- Credential stuffing across multiple accounts
- Distributed attacks against one account
- Possible account takeover from a new device, IP, or location

Documentation-only IP ranges are used, so the generated addresses do not represent real customers.


In [59]:
df =pd.read_csv(Path('../data/login_reject_history_3000.csv'))
df

,id,email,device_mac_id,ip,location,time_to_attempt,rejection_reason,is_suspicious,scenario
0,1,user9@example.com,02:74:94:28:77:33,192.0.2.98,"Dhaka, BD",2026-07-01T00:11:51Z,expired_session,0,normal_user_error
1,2,user9@example.com,02:74:94:28:77:33,192.0.2.98,"Dhaka, BD",2026-07-01T00:28:22Z,invalid_password,0,normal_user_error
2,3,user1@example.com,02:39:0C:8C:7D:72,192.0.2.36,"Dhaka, BD",2026-07-01T00:37:27Z,password_typo,0,normal_user_error
3,4,user9@example.com,02:74:94:28:77:33,192.0.2.98,"Dhaka, BD",2026-07-01T00:39:58Z,password_typo,0,normal_user_error
4,5,user8@example.com,02:96:B9:62:23:17,192.0.2.170,"Rajshahi, BD",2026-07-01T01:50:21Z,expired_session,0,normal_user_error
...,...,...,...,...,...,...,...,...,...
2995,2996,user10@example.com,02:A1:4D:E1:22:F0,192.0.2.91,"New York, US",2026-07-31T20:10:34Z,invalid_password,0,normal_user_error
2996,2997,user2@example.com,02:34:2C:D8:10:0F,192.0.2.24,"Chattogram, BD",2026-07-31T20:17:57Z,otp_failed,0,normal_user_error
2997,2998,user10@example.com,02:8E:E8:BA:53:BD,192.0.2.91,"Chattogram, BD",2026-07-31T22:47:08Z,password_typo,0,normal_user_error
2998,2999,user8@example.com,02:96:B9:62:23:17,192.0.2.170,"Rajshahi, BD",2026-07-31T22:49:41Z,password_typo,0,normal_user_error


In [50]:
'''
Time difference calculation and labeling 
sorted_by will take feature which will be identity to sort 
identity is indetity of entity to be group 
target , source featue of time stamp 
output_feature calculation from target 
time_diff
'''
class EntityHistoryFeatureTransformer(BaseEstimator, TransformerMixin):
      def __init__(
        self,
        sorted_by,
        identity,
        target,
        output_feature,
        time_diff
    ):
        self.sorted_by = sorted_by
        self.identity = identity
        self.target = target
        self.output_feature = output_feature
        self.time_diff = time_diff

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.sort_values(self.sorted_by).copy()

        previous_email = df.groupby(self.identity)[self.target].shift(1)

        df[self.output_feature] = (
        previous_email.notna()
        & previous_email.ne(df["email"])
        ).astype(int)

        return df

    def get_feature_names_out(self, input_features=None):
        return np.array([self.output_feature])

In [51]:
 
class PreviousValueChangeImputer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        sorted_by,
        identity,
        target,
        output_feature
    ):
        self.sorted_by = sorted_by
        self.identity = identity
        self.target = target
        self.output_feature = output_feature

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.sort_values("time_to_attempt").copy()

        previous_email = df.groupby("ip")["email"].shift(1)

        df["ip_email_changed"] = (
        previous_email.notna()
        & previous_email.ne(df["email"])
        ).astype(int)

        return df

    def get_feature_names_out(self, input_features=None):
        return np.array([self.output_feature])

# Missing-Value Rules for 22 Engineered Features

| Feature                              | Missing-value rule                                                    |
| ------------------------------------ | --------------------------------------------------------------------- |
| `seconds_since_email_last_attempt`   | Set to `2592000` seconds — 30 days                                    |
| `email_attempts_5m`                  | `0`                                                                   |
| `email_attempts_1h`                  | `0`                                                                   |
| `email_attempts_24h`                 | `0`                                                                   |
| `consecutive_email_failures`         | `0`                                                                   |
| `seconds_since_ip_last_attempt`      | Set to `2592000`                                                      |
| `ip_attempts_5m`                     | `0`                                                                   |
| `ip_attempts_1h`                     | `0`                                                                   |
| `ip_unique_emails_10m`               | `0`                                                                   |
| `ip_unique_emails_1h`                | `0`                                                                   |
| `device_unique_emails_1h`            | `0`                                                                   |
| `email_unique_ips_1h`                | `0`                                                                   |
| `email_unique_ips_24h`               | `0`                                                                   |
| `email_unique_devices_24h`           | `0`                                                                   |
| `email_unique_locations_24h`         | `0`                                                                   |
| `is_new_ip_for_email`                | `0` when the email has no previous history; otherwise calculate `0/1` |
| `is_new_device_for_email`            | `0` when the email has no previous history; otherwise calculate `0/1` |
| `is_new_location_for_email`          | `0` when the email has no previous history; otherwise calculate `0/1` |
| `location_changed_from_last_attempt` | `0` when no previous attempt exists                                   |
| `seconds_since_device_last_attempt`  | Set to `2592000`                                                      |
| `hour_sin`                           | Recalculate from `time_to_attempt`; do not statistically impute       |
| `hour_cos`                           | Recalculate from `time_to_attempt`; do not statistically impute       |




In [52]:
df

,id,email,device_mac_id,ip,location,time_to_attempt,rejection_reason,is_suspicious,scenario
0,1,user9@example.com,02:74:94:28:77:33,192.0.2.98,"Dhaka, BD",2026-07-01T00:11:51Z,expired_session,0,normal_user_error
1,2,user9@example.com,02:74:94:28:77:33,192.0.2.98,"Dhaka, BD",2026-07-01T00:28:22Z,invalid_password,0,normal_user_error
2,3,user1@example.com,02:39:0C:8C:7D:72,192.0.2.36,"Dhaka, BD",2026-07-01T00:37:27Z,password_typo,0,normal_user_error
3,4,user9@example.com,02:74:94:28:77:33,192.0.2.98,"Dhaka, BD",2026-07-01T00:39:58Z,password_typo,0,normal_user_error
4,5,user8@example.com,02:96:B9:62:23:17,192.0.2.170,"Rajshahi, BD",2026-07-01T01:50:21Z,expired_session,0,normal_user_error
...,...,...,...,...,...,...,...,...,...
2995,2996,user10@example.com,02:A1:4D:E1:22:F0,192.0.2.91,"New York, US",2026-07-31T20:10:34Z,invalid_password,0,normal_user_error
2996,2997,user2@example.com,02:34:2C:D8:10:0F,192.0.2.24,"Chattogram, BD",2026-07-31T20:17:57Z,otp_failed,0,normal_user_error
2997,2998,user10@example.com,02:8E:E8:BA:53:BD,192.0.2.91,"Chattogram, BD",2026-07-31T22:47:08Z,password_typo,0,normal_user_error
2998,2999,user8@example.com,02:96:B9:62:23:17,192.0.2.170,"Rajshahi, BD",2026-07-31T22:49:41Z,password_typo,0,normal_user_error


# Zero Imputer

In [53]:
zero_default_features = [
    "email_attempts_5m",
    "email_attempts_1h",
    "email_attempts_24h",
    "consecutive_email_failures",
    "ip_attempts_5m",
    "ip_attempts_1h",
    "ip_unique_emails_10m",
    "ip_unique_emails_1h",
    "device_unique_emails_1h",
    "email_unique_ips_1h",
    "email_unique_ips_24h",
    "email_unique_devices_24h",
    "email_unique_locations_24h",
    "is_new_ip_for_email",
    "is_new_device_for_email",
    "is_new_location_for_email",
    "location_changed_from_last_attempt"
]





df[zero_default_features]=np.nan
df[['seconds_since_email_last_attempt',
   'seconds_since_device_last_attempt',
   'is_new_location_for_email',
   'is_new_device_for_email',
   'is_new_ip_for_email',
   'location_changed_from_last_attempt']]=np.nan 






preprocessor = ColumnTransformer(
    transformers=[
        (
            "zero_imputer",
            SimpleImputer(strategy="constant",fill_value=0),
            zero_default_features
        ),
        
        (
             "month_imputer",
             SimpleImputer(strategy='constant',fill_value=2592000),
             ['seconds_since_email_last_attempt','seconds_since_device_last_attempt']
         ),
         (
             'history_feature_transformer',
              EntityHistoryFeatureTransformer(),
              ['is_new_location_for_email','is_new_device_for_email','is_new_ip_for_email' ]
         ),
         (
            'location_change',
             PreviousValueChangeImputer( sorted_by="time_to_attempt",
                 identity="email",
                 target="location",
                output_features="location_changed_from_last_attempt"),
            ["location_changed_from_last_attempt"]
         )        

    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

preprocessor.set_output(transform="pandas")

ColumnTransformer(remainder='passthrough',
                  transformers=[('zero_imputer',
                                 SimpleImputer(fill_value=0,
                                               strategy='constant'),
                                 ['email_attempts_5m', 'email_attempts_1h',
                                  'email_attempts_24h',
                                  'consecutive_email_failures',
                                  'ip_attempts_5m', 'ip_attempts_1h',
                                  'ip_unique_emails_10m', 'ip_unique_emails_1h',
                                  'device_unique_emails_1h',
                                  'email_unique_ips_1h', 'email_uniqu...,
                                  'email_unique_devices_24h',
                                  'email_unique_locations_24h',
                                  'is_new_ip_for_email',
                                  'is_new_device_for_email',
                                  'is_new_location_for_email',
                                  'location_changed_from_last_attempt']),
                                ('month_imputer',
                                 SimpleImputer(fill_value=2592000,
                                               strategy='constant'),
                                 ['seconds_since_email_last_attempt',
                                  'seconds_since_device_last_attempt'])],
                  verbose_feature_names_out=False)

In [54]:
final_df=preprocessor.fit_transform(df)
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 28 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   email_attempts_5m                   3000 non-null   float64
 1   email_attempts_1h                   3000 non-null   float64
 2   email_attempts_24h                  3000 non-null   float64
 3   consecutive_email_failures          3000 non-null   float64
 4   ip_attempts_5m                      3000 non-null   float64
 5   ip_attempts_1h                      3000 non-null   float64
 6   ip_unique_emails_10m                3000 non-null   float64
 7   ip_unique_emails_1h                 3000 non-null   float64
 8   device_unique_emails_1h             3000 non-null   float64
 9   email_unique_ips_1h                 3000 non-null   float64
 10  email_unique_ips_24h                3000 non-null   float64
 11  email_unique_devices_24h            3000 no

/Users/dawud/Documents/python/venv/lib/python3.9/site-packages/sklearn/impute/_base.py:572: FutureWarning: Currently, when `keep_empty_feature=False` and `strategy="constant"`, empty features are not dropped. This behaviour will change in version 1.8. Set `keep_empty_feature=True` to preserve this behaviour.
  warnings.warn(
/Users/dawud/Documents/python/venv/lib/python3.9/site-packages/sklearn/impute/_base.py:572: FutureWarning: Currently, when `keep_empty_feature=False` and `strategy="constant"`, empty features are not dropped. This behaviour will change in version 1.8. Set `keep_empty_feature=True` to preserve this behaviour.
  warnings.warn(


# is_new_ip_for_email
- step 1 : short by timestamp
- step 2 : curent dataset ip will find last ip's dataset
- step 3 : current dataset ip's email will compare last ip's data set
    - if no previous record found by the ip then 0
    - if previous record find by ip and current find by ip contain same email 0
    - if previous record find by ip and current find by ip contain different email 1

In [56]:
imputer =PreviousValueChangeImputer(sorted_by='time_to_attempt',identity='ip',target='email',output_feature='is_new_email_for_ip')
a_df =imputer.transform(final_df)
a_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3000 entries, 0 to 2999
Data columns (total 29 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   email_attempts_5m                   3000 non-null   float64
 1   email_attempts_1h                   3000 non-null   float64
 2   email_attempts_24h                  3000 non-null   float64
 3   consecutive_email_failures          3000 non-null   float64
 4   ip_attempts_5m                      3000 non-null   float64
 5   ip_attempts_1h                      3000 non-null   float64
 6   ip_unique_emails_10m                3000 non-null   float64
 7   ip_unique_emails_1h                 3000 non-null   float64
 8   device_unique_emails_1h             3000 non-null   float64
 9   email_unique_ips_1h                 3000 non-null   float64
 10  email_unique_ips_24h                3000 non-null   float64
 11  email_unique_devices_24h            3000 non-nul

## 4. Online-safe behavioral feature engineering

For each row, all rolling counts and novelty flags use **only earlier attempts**. This prevents future-data leakage during training and mirrors real-time inference.


In [37]:
def _seconds_since(current: pd.Timestamp, previous: pd.Timestamp | None) -> float:
    if previous is None:
        return 86400.0 * 30
    return max((current - previous).total_seconds(), 0.0)


def _prune_timestamp_queue(queue: deque[pd.Timestamp], cutoff: pd.Timestamp) -> None:
    while queue and queue[0] < cutoff:
        queue.popleft()


def _prune_pair_queue(queue: deque[tuple[pd.Timestamp, str]], cutoff: pd.Timestamp) -> None:
    while queue and queue[0][0] < cutoff:
        queue.popleft()


def add_behavior_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create online-safe features using only attempts earlier than each row."""
    missing = [col for col in BASE_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    work = df.copy()
    work["time_to_attempt"] = pd.to_datetime(work["time_to_attempt"], utc=True, errors="raise")
    work = work.sort_values(["time_to_attempt", "id"], kind="stable").reset_index(drop=True)

    email_times: dict[str, deque[pd.Timestamp]] = defaultdict(deque)
    ip_events: dict[str, deque[tuple[pd.Timestamp, str]]] = defaultdict(deque)
    email_ip_events: dict[str, deque[tuple[pd.Timestamp, str]]] = defaultdict(deque)
    email_device_events: dict[str, deque[tuple[pd.Timestamp, str]]] = defaultdict(deque)

    seen_ips: dict[str, set[str]] = defaultdict(set)
    seen_devices: dict[str, set[str]] = defaultdict(set)
    seen_locations: dict[str, set[str]] = defaultdict(set)
    last_email_time: dict[str, pd.Timestamp] = {}
    last_ip_time: dict[str, pd.Timestamp] = {}

    feature_rows: list[dict[str, Any]] = []

    for row in work.itertuples(index=False):
        timestamp = row.time_to_attempt
        email = str(row.email)
        ip = str(row.ip)
        device = str(row.device_mac_id)
        location = str(row.location)

        email_queue = email_times[email]
        ip_queue = ip_events[ip]
        email_ip_queue = email_ip_events[email]
        email_device_queue = email_device_events[email]

        _prune_timestamp_queue(email_queue, timestamp - pd.Timedelta(hours=24))
        _prune_pair_queue(ip_queue, timestamp - pd.Timedelta(hours=1))
        _prune_pair_queue(email_ip_queue, timestamp - pd.Timedelta(hours=1))
        _prune_pair_queue(email_device_queue, timestamp - pd.Timedelta(hours=24))

        attempts_email_5m = sum(t >= timestamp - pd.Timedelta(minutes=5) for t in email_queue)
        attempts_email_1h = sum(t >= timestamp - pd.Timedelta(hours=1) for t in email_queue)
        attempts_ip_5m = sum(t >= timestamp - pd.Timedelta(minutes=5) for t, _ in ip_queue)
        attempts_ip_1h = len(ip_queue)
        unique_emails_from_ip_10m = len(
            {value for t, value in ip_queue if t >= timestamp - pd.Timedelta(minutes=10)}
        )
        unique_ips_for_email_1h = len({value for _, value in email_ip_queue})
        unique_devices_for_email_24h = len({value for _, value in email_device_queue})

        hour = timestamp.hour + timestamp.minute / 60.0
        angle = 2.0 * math.pi * hour / 24.0
        feature_rows.append(
            {
                "attempts_email_5m": attempts_email_5m,
                "attempts_email_1h": attempts_email_1h,
                "attempts_ip_5m": attempts_ip_5m,
                "attempts_ip_1h": attempts_ip_1h,
                "unique_emails_from_ip_10m": unique_emails_from_ip_10m,
                "unique_ips_for_email_1h": unique_ips_for_email_1h,
                "unique_devices_for_email_24h": unique_devices_for_email_24h,
                "seconds_since_email_attempt": min(_seconds_since(timestamp, last_email_time.get(email)), 86400.0 * 30),
                "seconds_since_ip_attempt": min(_seconds_since(timestamp, last_ip_time.get(ip)), 86400.0 * 30),
                "is_new_ip_for_email": int(ip not in seen_ips[email]),
                "is_new_device_for_email": int(device not in seen_devices[email]),
                "is_new_location_for_email": int(location not in seen_locations[email]),
                "is_night": int(timestamp.hour < 5 or timestamp.hour >= 23),
                "is_weekend": int(timestamp.dayofweek >= 5),
                "hour_sin": math.sin(angle),
                "hour_cos": math.cos(angle),
            }
        )

        email_queue.append(timestamp)
        ip_queue.append((timestamp, email))
        email_ip_queue.append((timestamp, ip))
        email_device_queue.append((timestamp, device))
        seen_ips[email].add(ip)
        seen_devices[email].add(device)
        seen_locations[email].add(location)
        last_email_time[email] = timestamp
        last_ip_time[ip] = timestamp

    features = pd.DataFrame(feature_rows)
    enriched = pd.concat([work.reset_index(drop=True), features], axis=1)
    return enriched

## 5. Model construction and training

In [5]:
def _one_hot_encoder() -> OneHotEncoder:
    params: dict[str, Any] = {"handle_unknown": "ignore"}
    signature = inspect.signature(OneHotEncoder)
    if "sparse_output" in signature.parameters:
        params["sparse_output"] = True
    else:  # Compatibility with older scikit-learn versions.
        params["sparse"] = True
    return OneHotEncoder(**params)


def build_model() -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            ("categorical", _one_hot_encoder(), CATEGORICAL_FEATURES),
            ("numeric", StandardScaler(), NUMERIC_FEATURES),
        ],
        remainder="drop",
    )
    classifier = LogisticRegression(
        max_iter=2000,
        solver="liblinear",
        random_state=42,
    )
    return Pipeline([("preprocess", preprocessor), ("classifier", classifier)])


def _safe_auc(y_true: pd.Series, scores: np.ndarray) -> float | None:
    if len(set(y_true.tolist())) < 2:
        return None
    return float(roc_auc_score(y_true, scores))


def train_suspicious_attack_model(
    csv_path: str | Path,
    model_output_path: str | Path = "suspicious_login_model.joblib",
) -> dict[str, Any]:
    df = pd.read_csv(csv_path)
    if "is_suspicious" not in df.columns:
        raise ValueError("Training CSV must include is_suspicious target column")

    enriched = add_behavior_features(df)
    split_index = int(len(enriched) * 0.80)
    train_df = enriched.iloc[:split_index].copy()
    test_df = enriched.iloc[split_index:].copy()

    model = build_model()
    model.fit(train_df[MODEL_FEATURES], train_df["is_suspicious"].astype(int))

    probabilities = model.predict_proba(test_df[MODEL_FEATURES])[:, 1]
    predictions = (probabilities >= 0.50).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        test_df["is_suspicious"].astype(int), predictions, average="binary", zero_division=0
    )
    metrics = {
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "test_positive_rate": float(test_df["is_suspicious"].mean()),
        "roc_auc": _safe_auc(test_df["is_suspicious"], probabilities),
        "average_precision": float(average_precision_score(test_df["is_suspicious"], probabilities)),
        "precision_at_0_5": float(precision),
        "recall_at_0_5": float(recall),
        "f1_at_0_5": float(f1),
        "confusion_matrix_at_0_5": confusion_matrix(
            test_df["is_suspicious"].astype(int), predictions, labels=[0, 1]
        ).tolist(),
        "temporal_split_timestamp": str(test_df.iloc[0]["time_to_attempt"]),
    }

    bundle = {
        "model": model,
        "model_features": MODEL_FEATURES,
        "action_thresholds": ACTION_THRESHOLDS,
        "metrics": metrics,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "note": "Synthetic demonstration model; retrain and calibrate on real confirmed security outcomes.",
    }
    model_output_path = Path(model_output_path)
    model_output_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(bundle, model_output_path)
    return metrics

## 6. Hybrid threat scoring and security actions

The final score is:

```text
thret = 0.75 × model_probability + 0.25 × rule_score
```

Long or permanent blocks require repeated, strong evidence. A single unfamiliar device or location should normally trigger OTP rather than an irreversible block.


In [6]:
def calculate_rule_score(feature_row: pd.Series | dict[str, Any]) -> float:
    f = feature_row
    reason_weights = {
        "password_typo": 0.01,
        "expired_session": 0.02,
        "invalid_password": 0.05,
        "otp_failed": 0.12,
        "unknown_device": 0.13,
        "too_many_attempts": 0.20,
        "blocked_ip": 0.24,
    }

    score = 0.0
    score += min(float(f["attempts_email_5m"]) / 12.0, 1.0) * 0.17
    score += min(float(f["attempts_ip_5m"]) / 18.0, 1.0) * 0.17
    score += min(float(f["unique_emails_from_ip_10m"]) / 7.0, 1.0) * 0.21
    score += min(float(f["unique_ips_for_email_1h"]) / 6.0, 1.0) * 0.17
    score += min(float(f["unique_devices_for_email_24h"]) / 5.0, 1.0) * 0.08
    score += int(f["is_new_ip_for_email"]) * 0.05
    score += int(f["is_new_device_for_email"]) * 0.05
    score += int(f["is_new_location_for_email"]) * 0.04
    score += int(f["is_night"]) * 0.02
    score += reason_weights.get(str(f["rejection_reason"]), 0.04)
    return float(np.clip(score, 0.0, 1.0))


def recommend_action(score: float, feature_row: pd.Series | dict[str, Any]) -> str:
    f = feature_row
    repeated_abuse = (
        float(f["attempts_email_5m"]) >= 4
        or float(f["attempts_ip_5m"]) >= 6
        or float(f["unique_emails_from_ip_10m"]) >= 3
        or float(f["unique_ips_for_email_1h"]) >= 3
    )
    strong_abuse = (
        float(f["attempts_email_1h"]) >= 12
        or float(f["attempts_ip_1h"]) >= 18
        or float(f["unique_emails_from_ip_10m"]) >= 6
        or float(f["unique_ips_for_email_1h"]) >= 5
    )
    extreme_evidence = (
        float(f["attempts_ip_1h"]) >= 30
        or float(f["unique_emails_from_ip_10m"]) >= 8
        or (
            float(f["attempts_email_1h"]) >= 20
            and float(f["unique_ips_for_email_1h"]) >= 5
        )
    )

    # Long or permanent blocks require repeated abuse, not one unusual login.
    if score >= ACTION_THRESHOLDS["permanent_block"] and extreme_evidence:
        return "permanently block"
    if score >= ACTION_THRESHOLDS["temporary_block"] and strong_abuse:
        return "temporary block - contact help center"
    if score >= ACTION_THRESHOLDS["block_1_hour"] and strong_abuse:
        return "block for 1 hour"
    if score >= ACTION_THRESHOLDS["block_5_minutes"] and repeated_abuse:
        return "block for 5 minutes"
    if score >= ACTION_THRESHOLDS["otp_required"]:
        return "otp required"
    if score >= ACTION_THRESHOLDS["medium_risk"]:
        return "medium risk"
    return "low risk"


def explain_risk(feature_row: pd.Series | dict[str, Any]) -> list[str]:
    f = feature_row
    reasons: list[str] = []
    if float(f["attempts_email_5m"]) >= 5:
        reasons.append(f"{int(f['attempts_email_5m'])} earlier attempts for this email in 5 minutes")
    if float(f["attempts_ip_5m"]) >= 8:
        reasons.append(f"{int(f['attempts_ip_5m'])} earlier attempts from this IP in 5 minutes")
    if float(f["unique_emails_from_ip_10m"]) >= 4:
        reasons.append(
            f"this IP targeted {int(f['unique_emails_from_ip_10m'])} different emails in 10 minutes"
        )
    if float(f["unique_ips_for_email_1h"]) >= 3:
        reasons.append(f"this email was targeted from {int(f['unique_ips_for_email_1h'])} IPs in 1 hour")
    if int(f["is_new_device_for_email"]):
        reasons.append("new device for this email")
    if int(f["is_new_ip_for_email"]):
        reasons.append("new IP for this email")
    if int(f["is_new_location_for_email"]):
        reasons.append("new location for this email")
    if str(f["rejection_reason"]) in {"blocked_ip", "too_many_attempts", "otp_failed", "unknown_device"}:
        reasons.append(f"high-signal rejection reason: {f['rejection_reason']}")
    if not reasons:
        reasons.append("no strong repeated-attack pattern found")
    return reasons[:6]


def _hybrid_threat_score(model_probability: float, rule_score: float) -> float:
    # ML leads; deterministic controls retain influence for obvious security patterns.
    return float(np.clip(0.75 * model_probability + 0.25 * rule_score, 0.0, 1.0))


def score_history(
    csv_path: str | Path,
    model_path: str | Path,
    output_path: str | Path = "login_reject_history_scored.csv",
) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    enriched = add_behavior_features(df)
    bundle = joblib.load(model_path)
    model: Pipeline = bundle["model"]

    model_probabilities = model.predict_proba(enriched[MODEL_FEATURES])[:, 1]
    rule_scores = enriched.apply(calculate_rule_score, axis=1).to_numpy()
    threat_scores = np.clip(0.75 * model_probabilities + 0.25 * rule_scores, 0.0, 1.0)

    scored = df.copy()
    scored["thret"] = np.round(threat_scores, 6)
    scored["recommended_action"] = [
        recommend_action(float(score), enriched.iloc[index])
        for index, score in enumerate(threat_scores)
    ]

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    scored.to_csv(output_path, index=False)
    return scored


def assess_login_attempt(
    attempt: dict[str, Any],
    history: pd.DataFrame | str | Path,
    model_path: str | Path,
) -> dict[str, Any]:
    """Score one new rejected-login attempt and return the security action."""
    history_df = pd.read_csv(history) if isinstance(history, (str, Path)) else history.copy()
    required_without_id = [column for column in BASE_COLUMNS if column != "id"]
    missing = [column for column in required_without_id if column not in attempt]
    if missing:
        raise ValueError(f"Attempt is missing required fields: {missing}")

    new_id = str(attempt.get("id", f"new-attempt-{len(history_df) + 1}"))
    attempt_row = {column: attempt.get(column) for column in BASE_COLUMNS}
    attempt_row["id"] = new_id

    combined = pd.concat([history_df[BASE_COLUMNS], pd.DataFrame([attempt_row])], ignore_index=True)
    enriched = add_behavior_features(combined)
    selected = enriched[enriched["id"].astype(str) == new_id]
    if selected.empty:
        raise RuntimeError("Could not locate the new attempt after feature generation")
    feature_row = selected.iloc[-1]

    bundle = joblib.load(model_path)
    model: Pipeline = bundle["model"]
    model_probability = float(model.predict_proba(feature_row[MODEL_FEATURES].to_frame().T)[0, 1])
    rule_score = calculate_rule_score(feature_row)
    threat_score = _hybrid_threat_score(model_probability, rule_score)
    action = recommend_action(threat_score, feature_row)

    return {
        "id": new_id,
        "email": str(feature_row["email"]),
        "thret": round(threat_score, 6),
        "model_probability": round(model_probability, 6),
        "rule_score": round(rule_score, 6),
        "action": action,
        "reasons": explain_risk(feature_row),
        "behavior_features": {
            "attempts_email_5m": int(feature_row["attempts_email_5m"]),
            "attempts_email_1h": int(feature_row["attempts_email_1h"]),
            "attempts_ip_5m": int(feature_row["attempts_ip_5m"]),
            "attempts_ip_1h": int(feature_row["attempts_ip_1h"]),
            "unique_emails_from_ip_10m": int(feature_row["unique_emails_from_ip_10m"]),
            "unique_ips_for_email_1h": int(feature_row["unique_ips_for_email_1h"]),
            "unique_devices_for_email_24h": int(feature_row["unique_devices_for_email_24h"]),
            "is_new_ip_for_email": int(feature_row["is_new_ip_for_email"]),
            "is_new_device_for_email": int(feature_row["is_new_device_for_email"]),
            "is_new_location_for_email": int(feature_row["is_new_location_for_email"]),
        },
    }

## 7. Optional full-demo helper

In [7]:
def run_demo(output_dir: str | Path = ".") -> dict[str, Any]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    raw_csv = output_dir / "login_reject_history_3000.csv"
    model_file = output_dir / "suspicious_login_model.joblib"
    scored_csv = output_dir / "login_reject_history_scored.csv"

    generated = generate_dummy_file(raw_csv, n_rows=3000, random_state=42)
    metrics = train_suspicious_attack_model(raw_csv, model_file)
    scored = score_history(raw_csv, model_file, scored_csv)

    latest_time = pd.to_datetime(generated["time_to_attempt"], utc=True).max()
    example_attempt = {
        "email": "user1@example.com",
        "device_mac_id": "02:AA:BB:CC:DD:99",
        "ip": "203.0.113.250",
        "location": "London, GB",
        "time_to_attempt": (latest_time + pd.Timedelta(minutes=1)).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "rejection_reason": "unknown_device",
    }
    example_result = assess_login_attempt(example_attempt, generated, model_file)

    summary = {
        "raw_csv": str(raw_csv),
        "model_file": str(model_file),
        "scored_csv": str(scored_csv),
        "rows": int(len(generated)),
        "scored_rows": int(len(scored)),
        "metrics": metrics,
        "example_attempt": example_attempt,
        "example_result": example_result,
    }
    with open(output_dir / "demo_summary.json", "w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2)
    return summary

## 8. Generate the 3,000-row dummy dataset

In [4]:
OUTPUT_DIR = Path("suspicious_login_notebook_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_CSV = OUTPUT_DIR / "login_reject_history_3000.csv"
MODEL_FILE = OUTPUT_DIR / "suspicious_login_model.joblib"
SCORED_CSV = OUTPUT_DIR / "login_reject_history_scored.csv"

# df = generate_dummy_file(
#     output_path=RAW_CSV,
#     n_rows=3000,
#     random_state=42,
# )

print(f"Generated rows: {len(df):,}")
print(f"Saved to: {RAW_CSV.resolve()}")
display(df.head(10))


Generated rows: 3,000
Saved to: /Users/dawud/Documents/python/real-world-problem/cyber-security/suspicious_login_notebook_output/login_reject_history_3000.csv


,id,email,device_mac_id,ip,location,time_to_attempt,rejection_reason,is_suspicious,scenario
0,1,user9@example.com,02:74:94:28:77:33,192.0.2.98,"Dhaka, BD",2026-07-01T00:11:51Z,expired_session,0,normal_user_error
1,2,user9@example.com,02:74:94:28:77:33,192.0.2.98,"Dhaka, BD",2026-07-01T00:28:22Z,invalid_password,0,normal_user_error
2,3,user1@example.com,02:39:0C:8C:7D:72,192.0.2.36,"Dhaka, BD",2026-07-01T00:37:27Z,password_typo,0,normal_user_error
3,4,user9@example.com,02:74:94:28:77:33,192.0.2.98,"Dhaka, BD",2026-07-01T00:39:58Z,password_typo,0,normal_user_error
4,5,user8@example.com,02:96:B9:62:23:17,192.0.2.170,"Rajshahi, BD",2026-07-01T01:50:21Z,expired_session,0,normal_user_error
5,6,user10@example.com,02:8E:E8:BA:53:BD,192.0.2.91,"Chattogram, BD",2026-07-01T01:57:20Z,password_typo,0,normal_user_error
6,7,user7@example.com,02:16:EB:3F:C1:28,192.0.2.142,"Sylhet, BD",2026-07-01T01:58:13Z,expired_session,0,normal_user_error
7,8,user8@example.com,02:A5:DD:D0:8C:3E,192.0.2.120,"Dhaka, BD",2026-07-01T02:16:39Z,password_typo,0,normal_user_error
8,9,user8@example.com,02:96:B9:62:23:17,192.0.2.170,"Rajshahi, BD",2026-07-01T02:25:48Z,invalid_password,0,normal_user_error
9,10,user4@example.com,02:E5:8E:03:51:D8,192.0.2.88,"Rajshahi, BD",2026-07-01T02:33:34Z,expired_session,0,normal_user_error


### Dataset checks

In [ ]:
print("Columns:", df.columns.tolist())
print()
print("Suspicious-label distribution:")
display(df["is_suspicious"].value_counts().rename_axis("is_suspicious").to_frame("rows"))

print()
print("Scenario distribution:")
display(df["scenario"].value_counts().to_frame("rows"))


In [ ]:
df["is_suspicious"].value_counts().sort_index().plot(kind="bar")
plt.title("Synthetic suspicious-login target distribution")
plt.xlabel("is_suspicious")
plt.ylabel("Number of attempts")
plt.tight_layout()
plt.show()


## 9. Train the suspicious-attack model

In [ ]:
metrics = train_suspicious_attack_model(
    csv_path=RAW_CSV,
    model_output_path=MODEL_FILE,
)

print(f"Model saved to: {MODEL_FILE.resolve()}")
display(pd.Series(metrics, name="value").to_frame())


### Metric interpretation

Synthetic data can produce unusually high scores because attack patterns were deliberately generated. Treat these metrics as a software validation check, not evidence of real-world detection quality.


## 10. Score all historical attempts

In [ ]:
scored_df = score_history(
    csv_path=RAW_CSV,
    model_path=MODEL_FILE,
    output_path=SCORED_CSV,
)

print(f"Scored rows: {len(scored_df):,}")
print(f"Saved to: {SCORED_CSV.resolve()}")
display(scored_df.head(10))


In [ ]:
action_summary = (
    scored_df["recommended_action"]
    .value_counts()
    .rename_axis("recommended_action")
    .to_frame("rows")
)
display(action_summary)

scored_df["thret"].plot(kind="hist", bins=30)
plt.title("Distribution of hybrid threat scores")
plt.xlabel("thret")
plt.ylabel("Number of attempts")
plt.tight_layout()
plt.show()


## 11. Assess one new rejected-login attempt

Edit the `new_attempt` dictionary and rerun the cell. The required fields are:

- `email`
- `device_mac_id`
- `ip`
- `location`
- `time_to_attempt`
- `rejection_reason`


In [ ]:
latest_time = pd.to_datetime(df["time_to_attempt"], utc=True).max()

new_attempt = {
    "email": "user1@example.com",
    "device_mac_id": "02:AA:BB:CC:DD:99",
    "ip": "203.0.113.250",
    "location": "London, GB",
    "time_to_attempt": (latest_time + pd.Timedelta(minutes=1)).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "rejection_reason": "unknown_device",
}

result = assess_login_attempt(
    attempt=new_attempt,
    history=df,
    model_path=MODEL_FILE,
)

result


## 12. Reusable application method

In [ ]:
def check_rejected_login(attempt: dict[str, Any]) -> dict[str, Any]:
    """Application-facing wrapper for assessing one rejected login."""
    return assess_login_attempt(
        attempt=attempt,
        history=RAW_CSV,
        model_path=MODEL_FILE,
    )

# Example call
check_rejected_login(new_attempt)


## 13. Recommended production improvements

1. Replace synthetic labels with confirmed outcomes from security analysts or incident investigations.
2. Store successful-login history as well as rejected attempts to learn each user's normal devices, locations, and IP ranges.
3. Add device fingerprint, ASN, proxy/VPN/Tor signals, impossible travel, account age, and password-reset events.
4. Calibrate action thresholds using the financial and customer-support cost of false positives and false negatives.
5. Expire rolling history and retrain on a monitored schedule to address attacker behavior drift.
6. Require human review before permanent blocking unless a separate deterministic abuse policy is satisfied.
